Quantization: reduce the weight of the precision of the model

In [ ]:
%pip install -q transformers

In [ ]:
%pip install -q --upgrade bitsandbytes accelerate

In [1]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc
from dotenv import load_dotenv
import os

In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)

hf_token = os.getenv('HUGGING_FACE_CLAIRE')
if hf_token:
    print(f"Hugging Face Key exists and begins {hf_token[:8]}")
else:
    print(f"Hugging Face Key does not exists.")

#login into hugging face
login(token=hf_token, add_to_git_credential=True)


Hugging Face Key exists and begins hf_tnTKF


In [3]:
# LLAMA= "meta-llama/Llama-3.2-1B-Instruct"
LLAMA= "meta-llama/Meta-Llama-3.1-8B-Instruct"
PHI4 = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN2 = "Qwen/Qwen2-7B-Instruct"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
STARCODER2 = "bigcode/starcoder2-3b"

MIXTRAL = "mistralai/Mixtral-9x7b-Instruct-v0.1" #heavy

In [4]:
# messages = [
#     {"role": "system", "content": "You are a helpful assistant"},
#     {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
#   ]
messages = [   
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]

In [5]:
# checking internet conectivity:
import requests
print(requests.get("https://huggingface.co").status_code)

200


In [6]:
# Quantization Config - this allows us to load the model into memory and use less memory - reduce the precision

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [7]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token #End Of Sentence Token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

In [ ]:
# pytorch tensor = matrix
inputs

In [ ]:
# The model - create model, it connects to hugging face and download all the model weights and put in cache/memory, when disconnect it will be deleted

model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

In [ ]:
# then we see how much of memory it is occupying:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

### Looking under the hood at the Transformer model
The next cell prints the HuggingFace model object for Llama.

This model object is a Neural Network, implemented with the Python framework PyTorch. The Neural Network uses the architecture invented by Google scientists in 2017: the Transformer architecture.

While we're not going to go deep into the theory, this is an opportunity to get some intuition for what the Transformer actually is.

If you're completely new to Neural Networks, check out my [YouTube intro playlist](https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs) for the foundations.

Now take a look at the layers of the Neural Network that get printed in the next cell. Look out for this:

It consists of layers
There's something called "embedding" - this takes tokens and turns them into 4,096 dimensional vectors. We'll learn more about this in Week 5.
There are then 32 sets of groups of layers called "Decoder layers". Each Decoder layer contains three types of layer: (a) self-attention layers (b) multi-layer perceptron (MLP) layers (c) batch norm layers.
There is an LM Head layer at the end; this produces the output
Notice the mention that the model has been quantized to 4 bits.

It's not required to go any deeper into the theory at this point, but if you'd like to, I've asked our mutual friend to take this printout and make a tutorial to walk through each layer. This also looks at the dimensions at each point. If you're interested, work through this tutorial after running the next cell:

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

### Paramters (weights)
- Machine learning (training): train the model to give the best outcome for a given question
- Inference: predict outputs for inputs based on the data (parameters) they have
- Layering (Neural Network): pass the learning to another process, that pass to another process, and so on. It is called Neural Network because each layer is anaalogous to a human Neuron which is connected to others
- LLM (like chatGPT): 
    - Generative: 
        - inputs (known as prompts) converted into tokens
        - outputs: prediction of the most likely next token
    - Pre-trained:
        - The model has been given vast amounts of data from the internet, and paramenters (sliders) have been tweakd until it often predicts what comes next
    - Transformer: The model is a particular arragement of layers know as a Transformer. It is a "Neural Network Architecture".

- GPT has 10 times more the number of parameters. 
- It is so good predicting whta comes next that give us the impression it is intelligent and that it has memory
- memory is pretended by sending all the inputs you made in the conversation so the model checks all the conversation before answer

### "G" in GPT
- Generative
    - the model generate a probability of the possible answers then it return the most likely
    - when we ask for one of the other probability it generate anothe probability, and so on.
    - The inputs into the first layer are the sequence of tokens
    - The outputs of the last layer are the probabilities of every possible next token

    - Example for the input: *"How would you describe the color blue to someone who has never been able to see, in no more than 3 sentences"*
    <img src="NextTokenPrediction.png" width="300" height="300" style="display: block;" alt="Next token prediction"/>

- Pretrained
- Transformer

### Generalization
- Machine Learning: 
    - Ability to lean from data to make predictions in new situations not seen before without being explicitly programmed
- The ability to perform well on new examples outside the training data is called generallization
- Traditional machine learning depends os a good hypothesis on models and features
- Modern AI:
    - Deep Neural Networks learn patterns like "walk up penalty" example purely from examples
- How is ChatGPT so good? How is ChatGPT able to generalize so well?

### The "P" in GPT
- Generative
- Pretrained:
    - How it works:
    - 4 Steps, over and over again (100 billions times or more), through all the data:
        - Make a prediction (*Forward Pass*)
            - combine each token to each token

            <img src="forwardPass.png" width="300" height="300" style="display: block;" alt="Forward pass example/>

            - Then do it again because this example has 5 tokens, but this phase do it for each token by the number of tokens
            - In the end we calculate the probability og each generated result (like in the image "NextTokenPrediction.png")
            - Then we go to next step

        - How far off was it? (*Loss*)
            - How baddly we do? 
            - If the probability is 100 then loss should be zero. In the case of the example the correct answer was with 10% of chance to be choosen, what is wrong, so 90% of loss
            - The lower this probability. the higher the loss: loss = -log(probability) - *cross entrophy loss*
            - We don't want the model knows only this answer, we want it to learn about patterns to be able to generalize, so we go to next step

        - Wiggle parameters (*Backward Pass*)
            - it tweek each parameter:
                - Does the loss get better or worse as you wiggle each parameter?
                - Analyze the change in loss for a small change in parameters
                - The "gradients": make small change and apply that change to the mixers backward through calculations (The Chain Rule - Math techinque from High School)
        - Take a step in the right direction (*Optimization*)
            - Tiny change in the right direction to decrease the loss (trying to generalize) - "Gradient Descent"

    - Why it works:
    - Making it work even better

- Transformer

In [ ]:
# Execute this cell and look at what gets printed; investigate the layers

model

### And if you want to go even deeper into Transformers
In addition to looking at each of the layers in the model, you can actually look at the HuggingFace code that implements Llama using PyTorch.

Here is the HuggingFace Transformers repo:
https://github.com/huggingface/transformers

And within this, here is the code for Llama 4:
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

Obviously it's not neceesary at all to get into this detail - the job of an AI engineer is to select, optimize, fine-tune and apply LLMs rather than to code a transformer in PyTorch. OpenAI, Meta and other frontier labs spent millions building and training these models. But it's a fascinating rabbit hole if you're interested!

In [ ]:
# OK, with that, now let's run the model!
# send the request to the model with maximum og 80 tokens then get the response and decode this from machine language to human language

outputs = model.generate(inputs, max_new_tokens=80)
print(tokenizer.decode(outputs[0]))

In [ ]:
# Clean up memory
# Thank you Kuan L. for helping me get this to properly free up memory!
# If you select "Show Resources" on the top right to see GPU memory, it might not drop down right away
# But it does seem that the memory is available for use by new models in the later code.

del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

I'm using a HuggingFace utility called TextStreamer so that results stream back. To stream results, we simply replace:
outputs = model.generate(inputs, max_new_tokens=80)
With:
streamer = TextStreamer(tokenizer)
outputs = model.generate(inputs, max_new_tokens=80, streamer=streamer)

Also I've added the argument add_generation_prompt=True to my call to create the Chat template. This ensures that Phi generates a response to the question, instead of just predicting how the user prompt continues. Try experimenting with setting this to False to see what happens. You can read about this argument here:

https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts

In [ ]:
def generate(model, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)

In [ ]:
generate(PHI4, messages)

In [ ]:
%env CUDA_LAUNCH_BLOCKING=1


In [ ]:
# gemma doesn't support system message so we redefine message
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
generate(GEMMA, messages, quant=False)

In [13]:
generate(QWEN, messages)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

<|im_start|>user
Tell a light-hearted joke for a room of Data Scientists<|im_end|>
<|im_start|>assistant
Sure! Here's a light-hearted, data-savvy joke just for the Data Scientists in the room:

---

Why did the data scientist break up with the correlation coefficient?

Because it was always *highly* dependent on the variable — and never really *independent* of the relationship!

😄

(And yes, they still have a great *scatter plot* of their past — just not


In [14]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)

tokenizer_config.json: 0.00B [00:00, ?B/s]

d:\AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\iarar\.cache\huggingface\hub\models--deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<｜begin▁of▁sentence｜><｜User｜>Tell a light-hearted joke for a room of Data Scientists<｜Assistant｜><think>
Alright, I need to come up with a light-hearted joke for a room of data scientists. Let's see, the key here is to make it witty, funny, and not too technical since they're all data scientists. Maybe I can play on the idea of a "data party," which is a common setting for such conversations.

Hmm, what's a common joke about data? Maybe something about data being a "data set" that can be manipulated. But that might be too serious. Alternatively, I could think of a situation where data scientists are working together, maybe they're trying to solve a problem or make a decision. 

Wait, the user already provided a joke: "Why don't scientists trust data scientists? Because they always bring up the p-value!" That's already quite good. It's a classic joke about p-values in statistics, which is a key concept in data science. It's simple, relatable, and uses a common statistical term in a humo